# Back to Basics, sklearn edition — the three middle acts in the demo's dialect

Same story, same order, same 🎤 stage boxes as `back-to-basics.ipynb` — but every model
is fit the way the **demo** fits it: sklearn idioms, `Pipeline`s, `cross_val_score`.
Read the numpy twin to see the gears; run this one to rehearse the dialect you'll
actually present.

> ⚙ **This notebook needs your local sklearn env** (the repo has used sklearn 1.9 /
> py3.13 at `/opt/miniconda3`). **Run All takes ~3–5 min** (the MI cell is the slow
> one). Expected values are annotated before each cell — sanity-check as you go:
> **verified** = previously computed and logged in the briefings;
> **expect ≈** = the numpy twin's executed result, which this should land near.

> ⚠️ **Act numbering check**, once more: `slides/talk_outline.md` has
> **Act II = feature selection, Act III = regularization, Act IV = tuning/model
> selection.** This notebook follows the outline's order.

## The one story

22,000 incident reports, one question per report: *what hazard is this?*

> **The training data contains signal and noise, and the model cannot tell them apart.**
> It will fit both, enthusiastically. Your three acts are three tools for one job —
> making sure it fits the signal.

**Act II** scores columns *before* the model sees them (dependence with the class,
measured by chi²/MI). **Act III** leashes the model so it can't lean on noise — and the
L1 leash *performs selection as a side effect* (the hinge between acts). **Act IV** is
the evidence discipline: macro-F1, cross-validation, the one-SE rule, bake-offs on
shared folds — and the leakage commandment that makes it all valid.

LLM translations stay taped to the wall: noise-fitting = memorization (31 params/char in
the class's GPT run), the leash = weight decay/dropout, the evidence = `estimate_loss`'s
train/val split.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, time
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2, mutual_info_classif
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, classification_report
import sklearn
plt.rcParams.update({'figure.dpi': 110, 'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False})
print('sklearn', sklearn.__version__)

# ---- load CPSC, keep the top-12 hazard classes ----
df = pd.read_csv('../data/cpsc_merged.csv')
df = df[df.product_1_hazard.notna() & (df.product_1_hazard.str.strip() != '')
        & df.incident_description.notna()]
top12 = df.product_1_hazard.value_counts().nlargest(12).index
df = df[df.product_1_hazard.isin(top12)].reset_index(drop=True)
texts, y_lab = df.incident_description.str.lower(), df.product_1_hazard
print(f'\n{len(df)} reports, 12 classes    <- expect 22,087')
print(y_lab.value_counts().to_string())
print(f'\nimbalance {y_lab.value_counts().iloc[0]}:{y_lab.value_counts().iloc[-1]}'
      f' = {y_lab.value_counts().iloc[0]/y_lab.value_counts().iloc[-1]:.1f}:1   <- expect 15.7:1')

In [ ]:
# ---- two representations, used deliberately ----
# 1) binary presence counts (Acts II-III teaching: chi2's natural habitat)
binvec = CountVectorizer(binary=True, lowercase=True, max_features=1000,
                         token_pattern=r'(?u)\b[a-z]{2,}\b')
Xbin = binvec.fit_transform(texts)                     # 22k x 1000 sparse 0/1
vocab = np.array(binvec.get_feature_names_out())
w2j = {w: j for j, w in enumerate(vocab)}
print('presence matrix:', Xbin.shape)

# 2) raw text, vectorized INSIDE a Pipeline (Act IV: the leakage-proof idiom)
#    -- built later, where it belongs.

y = y_lab.values
# one stratified 80/20 split; Act IV explains why 'stratify' (and then does better)
Xtr, Xte, ytr, yte, itr, ite = train_test_split(
    Xbin, y, np.arange(len(y)), test_size=0.2, stratify=y, random_state=0)
print(f'train {Xtr.shape[0]} / test {Xte.shape[0]}, class proportions preserved')

---
# Part 1 · Feature selection (Act II)

**What does "informative" mean?** A column is informative about the class exactly when
they are **statistically dependent** — when seeing the column should change your belief
about the class. A column independent of the class is provably useless to any
classifier. Every filter method measures, per column, *distance from independence*.
One real word first.

*(Expect ≈: "fire" in ~28% of reports; 68% of them are Thermal-Fire vs 2% without —
a ~31× belief shift. Verified in the workbook: 4,237 vs 344 raw counts.)*

In [ ]:
fire = np.asarray(Xbin[:, w2j['fire']].todense()).ravel().astype(bool)
tab = pd.crosstab(fire, y_lab == 'Thermal - Fire',
                  rownames=['"fire" in text'], colnames=['Thermal-Fire hazard'])
print(tab, '\n')
p1 = tab.loc[True, True] / tab.loc[True].sum()
p0 = tab.loc[False, True] / tab.loc[False].sum()
print(f'P(Thermal-Fire | "fire" present) = {p1:.0%}')
print(f'P(Thermal-Fire | "fire" absent)  = {p0:.0%}')
print(f'belief multiplier ~{p1/p0:.0f}x. That shift IS dependence; now we score it.')

## Ruler #1 — chi²

Under independence the contingency table would equal the product of its margins:
$E_{wc} = (\text{row total})_w(\text{col total})_c/n$. chi² totals the squared,
variance-scaled disagreement $\sum (O-E)^2/E$. Model-free — that's what *filter* means.
sklearn: `chi2(X, y)` returns `(scores, pvalues)` per column, vectorized.

*(Expect ≈ top of the ranking: `monoxide`, `carbon`, `fire`, `pool`, `drowning`,
`fell`… — rare-but-decisive words score huge. Fine print: sklearn's `chi2` scores the
word-**present** row only — a one-sided variant of the full 2×K table the numpy twin
builds — so exact ranks may shuffle slightly between the two. Same idea, same winners.)*

In [ ]:
chi_scores, chi_p = chi2(Xbin, y)
rank_chi = np.argsort(-chi_scores)
print('chi^2 top 15 of 1,000 words:')
for r, j in enumerate(rank_chi[:15], 1):
    print(f'  {r:2d}. {vocab[j]:14s} {chi_scores[j]:9.0f}')
print('\nbottom 3 (words that know nothing about hazards):')
for j in rank_chi[-3:]:
    print(f'      {vocab[j]:14s} {chi_scores[j]:9.1f}')

## Ruler #2 — mutual information

Same table, different question: *how many bits does the word buy me about the class?*
$I(W;C) = H(W) + H(C) - H(W,C)$, and crucially $I(W;C) \le H(W)$: a rare word has a low
ceiling **no matter how decisive it is when present**. chi² has no cap — so the two
rulers *must* disagree about rare-but-decisive words.

> **The Briefing-1 MI trap, in the flesh:** `mutual_info_classif` on sparse *TF-IDF*
> floats silently treats them as categories (garbage: stopwords rank #1). Our columns
> are genuinely discrete (0/1 presence), so `discrete_features=True` is *correct* here.
> On TF-IDF you'd need dense input + `discrete_features=False` (kNN estimator).

*(Expect ≈: `fire` MI #1 but chi² #3; `monoxide` chi² #1 but MI ~#16. Verified on the
full 6,907-word TF-IDF vocabulary: monoxide falls to MI #142. This cell is the slow one
— on the order of a minute.)*

In [ ]:
t0 = time.time()
mi = mutual_info_classif(Xbin.toarray(), y, discrete_features=True, random_state=0)
rank_mi = np.argsort(-mi)
print(f'({time.time()-t0:.0f}s)  MI top 10:')
for r, j in enumerate(rank_mi[:10], 1):
    print(f'  {r:2d}. {vocab[j]:14s} {mi[j]:.4f} nats')

def where(word, order):
    return int(np.where(order == w2j[word])[0][0]) + 1
for w in ['fire', 'monoxide']:
    pw = Xbin[:, w2j[w]].mean()
    Hw = -(pw*np.log(pw) + (1-pw)*np.log(1-pw))
    print(f"\n{w!r}: chi2 #{where(w, rank_chi)}, MI #{where(w, rank_mi)} | "
          f"in {pw:.1%} of reports | MI ceiling H(w) = {Hw:.3f} nats")

**Read the disagreement — the depth marker of Act II.** `monoxide` is nearly a perfect
label when present, so chi² adores it; but its MI ceiling is tiny, so MI demotes it.
`fire` is common *and* reliable — MI's favorite.

> **chi² ≈ association strength. MI ≈ association × frequency.** Neither is wrong.
> (Theorem-grade version for backup slides: $G^2 = 2N\cdot\text{MI}$ exactly; Pearson's
> χ² is its Taylor approximation, and they part ways precisely on rare, extreme words.)

*Wrapper* methods (RFE): retrain per subset, combinatorially brutal — mention, skip.
*Embedded*: the model selects while training — needs Act III, leave it hanging.

## Does selection work? (still slightly dishonest)

`SelectKBest` fit on the **training split only** — never let test rows vote on which
features survive (Act IV's finale shows the crime scene).

*(Expect ≈ the numpy twin's pattern: all-1000 ≈ .82, top-300 ≈ .81, top-100 ≈ .77,
random-100 ≈ .36 macro-F1. Verified on the full TF-IDF demo: 6,907 → 1,000 features
costs 0.001 macro-F1: 0.8562 → 0.8552.)*

In [ ]:
clf = lambda: LogisticRegression(max_iter=2000, random_state=0)
rows = {}
for name, k in [('all 1,000 words', 1000), ('top 300 by chi2', 300), ('top 100 by chi2', 100)]:
    sel = SelectKBest(chi2, k=k).fit(Xtr, ytr)          # scored on TRAIN only
    m = clf().fit(sel.transform(Xtr), ytr)
    rows[name] = f1_score(yte, m.predict(sel.transform(Xte)), average='macro')

f1r = []
for s in range(5):
    cols = np.random.default_rng(s).choice(1000, 100, replace=False)
    m = clf().fit(Xtr[:, cols], ytr)
    f1r.append(f1_score(yte, m.predict(Xte[:, cols]), average='macro'))
rows['100 RANDOM words'] = float(np.mean(f1r))

print(f'{"feature set":24s}  test macro-F1')
for k_, v in rows.items():
    print(f'  {k_:24s}  {v:.3f}')

**Read it:** (1) chosen ≫ random at the same budget — that gap *is* selection;
(2) a fraction of the columns ≈ the full set — most vocabulary is noise or redundancy;
(3) push too far and **macro-F1 bleeds before accuracy** — rare classes lose their few
signal words first (why this talk scores macro-F1).

Two sins remain, both deliberate: `k` and the classifier's `C` were chosen by hand, and
judged on one split. Act IV removes every "trust me."

> **🎤 Act II, what you say on stage:** *"A feature is informative exactly when it's
> statistically dependent on the class — so we can score every column before fitting
> anything. Two rulers: chi² rewards rare-but-decisive words like `monoxide`; mutual
> information rewards common-and-reliable ones like `fire`. They disagree, and the
> disagreement is informative. With scores in hand, selection is nearly free: 1,000 of
> 6,907 TF-IDF features cost 0.001 macro-F1. One family is missing: what if the model
> itself could select, while it trains? For that, it needs a leash — next act."*

---
# Part 2 · Regularization (Act III)

## See the disease

The disease is **variance**: given freedom, a model fits *this* sample's noise, so a
fresh sample would yield a wildly different model. Deep-learning name: memorization
(the class's GPT: ~31 params per training character — it memorized the article).
Smallest possible lab: 12 points, polynomial degree as the freedom knob — and watch
**how** the flexible fit cheats: gigantic coefficients in delicate cancellation.

*(Expect ≈: degree-11 max |coef| in the tens of thousands; the ridge-tamed overlay sane.)*

In [ ]:
r2 = np.random.default_rng(3)
xs = np.sort(r2.uniform(-1, 1, 12)).reshape(-1, 1)
truth = lambda x: np.sin(2.5 * x)
yn = truth(xs).ravel() + r2.normal(0, .25, 12)
grid = np.linspace(-1.02, 1.02, 400).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.3), sharey=True)
for ax, deg in zip(axes, [1, 4, 11]):
    pf = PolynomialFeatures(deg, include_bias=False)
    lr = LinearRegression().fit(pf.fit_transform(xs), yn)
    ax.plot(grid, lr.predict(pf.transform(grid)), lw=2, label='fit')
    ax.plot(grid, truth(grid), '--', color='gray', lw=1, label='truth')
    ax.scatter(xs, yn, zorder=3, s=25, color='k')
    if deg == 11:                                    # the leash, previewed
        rr = Ridge(alpha=1e-3).fit(pf.fit_transform(xs), yn)
        ax.plot(grid, rr.predict(pf.transform(grid)), lw=2, ls='-.',
                label='same degree + tiny ridge')
    ax.set_ylim(-2.2, 2.2)
    ax.set_title(f'degree {deg} | max|coef| = {np.abs(lr.coef_).max():,.0f}')
axes[0].legend(loc='lower right', fontsize=8); axes[2].legend(loc='lower right', fontsize=8)
plt.tight_layout(); plt.show()

Degree 1 is too stiff (**bias**); degree 11 threads every point and hallucinates
between them (**variance**) — using coefficients in the tens of thousands that nearly
cancel. If noise-fitting *requires* huge fragile coefficients, **penalizing coefficient
size forbids noise-fitting**:

$$\hat\beta_\lambda=\arg\min_\beta\;\lVert y-X\beta\rVert_2^2+\lambda\lVert\beta\rVert_2^2\;\;(\text{ridge})\qquad\text{or}\qquad+\lambda\lVert\beta\rVert_1\;\;(\text{lasso})$$

sklearn naming, worth 10 seconds on stage: `Ridge(alpha=λ)` uses λ directly;
`LogisticRegression(C=1/λ)` uses the **inverse** — big C = weak leash — **and it
regularizes by default** (`penalty="l2", C=1.0`): everyone in the course who "never used
regularization" already did. Weight decay in AdamW is the same idea in optimizer form
(decoupling caveat: crash course §9).

## The ledger: MSE = bias² + variance, exactly

*(Expect ≈: identity holds to ~1e-15; U-shaped MSE with an interior minimum well below
the OLS end. Verified draw in Briefing 2: minimum 2.44 vs 4.56 near-OLS — about half.)*

In [ ]:
r3 = np.random.default_rng(1)
n_, p_ = 60, 30
Xs = r3.normal(size=(n_, p_))
beta = np.zeros(p_); beta[:5] = [2., -1.5, 1., 3., -2.]
f_true, sig = Xs @ beta, 2.0
lams = np.array([.001, .5, 2, 8, 32, 128, 512])

E = np.zeros((len(lams), 400, p_))
for b_ in range(400):
    yb_ = f_true + r3.normal(0, sig, n_)
    for li, lam_ in enumerate(lams):
        E[li, b_] = Ridge(alpha=lam_, fit_intercept=False).fit(Xs, yb_).coef_

print(f'{"lambda":>8} {"bias^2":>8} {"var":>8} {"sum":>8} {"MC MSE":>8}')
b2s, vs, ms = [], [], []
for li, lam_ in enumerate(lams):
    m = E[li].mean(0)
    b2 = ((m - beta)**2).sum(); v = E[li].var(0).sum()
    mse = ((E[li] - beta)**2).sum(1).mean()
    b2s.append(b2); vs.append(v); ms.append(mse)
    print(f'{lam_:8.3f} {b2:8.3f} {v:8.3f} {b2+v:8.3f} {mse:8.3f}')
print(f'identity: max|b2+v-MSE| = {max(abs(b+v-m_) for b,v,m_ in zip(b2s,vs,ms)):.1e}')

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(lams, b2s, 'o-', label='bias$^2$ (you bought it)')
ax.plot(lams, vs, 's-', label='variance (you sold it)')
ax.plot(lams, ms, 'k^-', lw=2, label='MSE = the sum (a U)')
ax.set_xscale('log'); ax.set_xlabel('$\\lambda$'); ax.set_ylabel('error')
ax.legend(fontsize=9); ax.set_title('The bias-variance ledger, measured')
plt.tight_layout(); plt.show()

**Read the ledger.** Bias² only rises, variance only falls, the sum is a **U with an
interior minimum** — some leash beats none, measurably. Every regularizer you'll ever
meet (λ, `k`, `max_iters`, dropout rate) walks this same U. *(Simplification marker:
classical regime, $p$ ~ $n$ — where the demo lives. Overparameterized deep nets add
double descent; Q&A armor, not stage material.)*

**Where ridge aims** (numpy twin does the full SVD demo): along each principal direction
the coefficient is multiplied by $s_j = d_j^2/(d_j^2+\lambda)$, while OLS variance blows
up as $\sigma^2/d_j^2$ — so ridge **shrinks hardest exactly where the estimate is
trash** (twin's run: $s_j$ = 0.97 on a strong direction vs **0.022** on a near-collinear
one at λ=5), and across 200 bootstrap worlds the collinear pair's OLS coefficients
swung with std ≈ 2.1 while ridge held them at ≈ 0.07. Variance-*targeted* humility.
Bayes read: Gaussian prior, MAP estimate.

## Lasso: the leash that amputates

L2's pull weakens near zero (derivative $2\lambda\beta$), so it never arrives; L1's pull
is **constant** ($\pm\lambda$), so small coefficients hit *exactly* zero — the
soft-threshold "subtract & clamp." Corners of the L1 ball sit on axes. **Embedded
selection = Act II's job performed by the optimizer.** The real dial, in sklearn:

*(Verified in Briefing 1 with this exact API: `C=0.05` → 12 words, F1 ≈ 0.72;
`C=1.0` → 278 words, F1 ≈ 0.87. Note C = 1/λ: the dial runs* **backwards** *relative to
the numpy twin's λ. `random_state=0` pinned — liblinear is stochastic otherwise.)*

In [ ]:
yf = (y == 'Thermal - Fire')
Xf_tr, Xf_te, yf_tr, yf_te = Xbin[itr], Xbin[ite], yf[itr], yf[ite]

print(f'{"C (=1/lambda)":>14} {"words kept":>11} {"test F1":>8}   words (when few)')
for C_ in [0.01, 0.05, 0.2, 1.0]:
    m = LogisticRegression(penalty='l1', solver='liblinear', C=C_,
                           random_state=0).fit(Xf_tr, yf_tr)
    w = m.coef_.ravel()
    keep = np.where(np.abs(w) > 1e-10)[0]
    f1_ = f1_score(yf_te, m.predict(Xf_te))
    names = ', '.join(vocab[j] for j in keep[np.argsort(-np.abs(w[keep]))][:7])
    print(f'{C_:14.2f} {len(keep):11d} {f1_:8.3f}   {names if len(keep) <= 7 else names + ", ..."}')

m = LogisticRegression(penalty='l1', solver='liblinear', C=1.0,
                       random_state=0).fit(Xf_tr, yf_tr)
w = m.coef_.ravel()
print('\nmost NEGATIVE weights at C=1.0:', ', '.join(vocab[j] for j in np.argsort(w)[:5]))

**Read the dial — this is a slide.** Tighten the leash and the model keeps a tiny,
*optimizer-chosen* vocabulary that already performs; loosen it and words are bought back
in evidence order. And the **negative weights** (`electrical`, `cord`, `pool`…) are the
model finding class *boundaries* — words that argue against fire by voting for the
electrical and submersion hazards.

**Dropout & early stopping, one breath each, then the GPT callback:** dropout = a
stochastic capacity penalty (≈ adaptive ridge for GLMs); early stopping bounds how far
training travels toward the noise — `max_iters` walks the same U as λ. So
`gpt_train.py` regularizes three ways (dropout 0.1, AdamW weight decay, finite
iterations) — and needs to, at ~31 params/char; the CPSC re-run (0.127 params/char,
240× the data pressure) barely does. **Regularization matters exactly when parameters
outrun data.** Watch that literally:

*(Expect ≈ the twin's executed result: full data — train barely prefers tiny λ, test
peaks gently ≈ 0.84. 1,200 rows — train hits a perfect 1.000 at tiny λ while test
collapses to ≈ 0.32, and α=10 rescues test to ≈ 0.83.)*

In [ ]:
def vote_table(idx_rows, title):
    Xa, ya = Xbin[idx_rows], y[idx_rows]
    Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(Xa, ya, test_size=0.25,
                                                  stratify=ya, random_state=42)
    print(title)
    print(f'{"alpha":>10} {"TRAIN macro-F1":>15} {"TEST macro-F1":>15}')
    for a_ in [1e-3, 1e-1, 1e1, 1e3]:
        m = RidgeClassifier(alpha=a_).fit(Xa_tr, ya_tr)
        print(f'{a_:10.0e} {f1_score(ya_tr, m.predict(Xa_tr), average="macro"):15.3f} '
              f'{f1_score(ya_te, m.predict(Xa_te), average="macro"):15.3f}')
    print()

vote_table(np.arange(len(y)), 'ALL 22k rows (data comfortably outnumbers 1,000 features):')

r8 = np.random.default_rng(11)
small = np.concatenate([r8.choice(np.where(y == c)[0], 100, replace=False)
                        for c in np.unique(y)])
vote_table(small, 'ONLY 1,200 rows (parameters outrun data -- the GPT regime):')

print('Both times TRAIN votes alpha -> 0 (freedom fits its own noise); TEST disagrees,')
print('loudly when data is scarce. Every knob needs a judge that is not the training set.')

> **🎤 Act III, what you say on stage:** *"A flexible model fits noise using huge
> coefficients that delicately cancel — so penalize coefficient size. That trades a
> little bias for a lot of variance, and the sum is a U: some leash is better than none,
> we measured it. Ridge shrinks hardest exactly where the data determines things worst.
> And the L1 leash does something magic: it drags weak coefficients to exactly zero —
> the model performs feature selection while it trains: a handful of optimizer-chosen
> words already detect fire. You've all used regularization twice already: sklearn's
> LogisticRegression penalizes by default — and C is one-over-λ, inverse! — and
> `gpt_train.py`'s dropout and AdamW are the same bet. One question remains: who picks
> λ? Not the training loss — it always votes for maximum freedom. Watch."*

---
# Part 3 · Tuning & model selection (Act IV)

Four pieces remove every "trust me" from Acts II–III: a **metric** that matches the
mission (macro-F1 — 15.7:1 imbalance), an honest **estimator** of out-of-sample skill
(CV), a **decision rule** for knobs (one-SE), and a **comparison protocol** (same folds,
same metric, SEs shown). Then the horror story the whole apparatus exists to prevent.

## The metric, in four printed numbers

*(Verified: accuracy 0.229 vs macro-F1 0.031. Deep dive: imbalance crash course.)*

In [ ]:
dummy = DummyClassifier(strategy='most_frequent').fit(Xtr, ytr)
pred = dummy.predict(Xte)
print(f'the "always predict {pd.Series(ytr).mode()[0]}" model:')
print(f'  accuracy: {(pred == yte).mean():.3f}     <- looks like a working model')
print(f'  macro-F1: {f1_score(yte, pred, average="macro"):.3f}     <- tells the truth')
print('\nEleven of twelve classes get F1 = 0, and macro-F1 says so:')
print(classification_report(yte, pred, zero_division=0, digits=2).splitlines()[-4])

## Cross-validation is a for-loop (that sklearn writes for you)

Five stratified folds; five times train-on-four, judge-on-the-fifth; every row judged
once by a model that never saw it — and five opinions give you a **standard error**, so
you know when a difference is noise. `cross_val_score` / `GridSearchCV` do exactly this,
with `StratifiedKFold` automatic for classifiers. LLM read: `estimate_loss` is 1-fold
CV; LLMs afford one val set because data is oceanic — at 22k rows we recycle.

*(Expect ≈ twin: a plateau ≈ 0.827 across α ∈ [0.01, 10], cliff by 10³–10⁴, one-SE pick
α = 10. Verified on the briefing's setup: plateau 0.809, one-SE λ = 10.)*

In [ ]:
alphas = np.array([1e-2, 1e-1, 1e0, 1e1, 1e2, 1e3, 1e4])
gs = GridSearchCV(RidgeClassifier(), {'alpha': alphas},
                  scoring='f1_macro', cv=5, n_jobs=-1).fit(Xbin, y)
mean = gs.cv_results_['mean_test_score']
se   = gs.cv_results_['std_test_score'] / np.sqrt(5)     # std over folds -> SE

best = int(np.argmax(mean))
one_se = max(i for i in range(len(alphas)) if mean[i] >= mean[best] - se[best])
print(f'{"alpha":>10} {"CV macro-F1":>12} {"+/- SE":>8}')
for i, a_ in enumerate(alphas):
    tag = '  <- best' if i == best else ('  <- one-SE pick' if i == one_se else '')
    print(f'{a_:10.0e} {mean[i]:12.3f} {se[i]:8.3f}{tag}')

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.errorbar(alphas, mean, yerr=se, fmt='o-', capsize=3)
ax.axhline(mean[best] - se[best], ls=':', color='gray', label='best minus one SE')
ax.plot(alphas[one_se], mean[one_se], 'r*', ms=16, label=f'one-SE pick: $\\alpha$={alphas[one_se]:g}')
ax.set_xscale('log'); ax.set_xlabel('$\\alpha$ (= $\\lambda$)'); ax.set_ylabel('CV macro-F1')
ax.set_title('The tuning curve: plateau, cliff, principled pick')
ax.legend(fontsize=9); plt.tight_layout(); plt.show()

**Read the curve:** a **plateau** (the knob matters at the edges — coarse grids are
fine, and when only a few of many knobs matter, random search beats grid: verified
P = 0.877 at budget 9); a **cliff** (the U's right arm); and the **one-SE rule** —
the best point wins by less than its own error bar, so take the most regularized model
within one SE. The λ = 10 that Parts 1–2 took on faith: debt paid, on evidence.

## The bake-off — the demo's actual four model types

Same folds, same metric, full `Pipeline`s on **raw text** (vectorizer and selector
inside — leakage-proof by construction). This cell **rehearses the demo's centerpiece.**

*(Verified reference: the tuned TF-IDF + logistic pipeline reaches ≈ 0.856 macro-F1
(Briefing 1); the twin's presence-feature bake-off ran majority .031 / centroid .662 /
ComplementNB .707 / ridge .827. Expect the pipelines below in the .78–.86 band, tree
lower; ~1–2 min.)*

In [ ]:
def pipe(clf_):
    return Pipeline([('tfidf', TfidfVectorizer(min_df=5)),      # no english stop list (fire!)
                     ('select', SelectKBest(chi2, k=1000)),
                     ('clf', clf_)])

contenders = {
    'majority (Dummy)':  DummyClassifier(strategy='most_frequent'),
    'ComplementNB':      ComplementNB(),
    'LinearSVC':         LinearSVC(random_state=0),
    'LogisticRegression (balanced)': LogisticRegression(max_iter=2000, class_weight='balanced',
                                                        random_state=0),
    'DecisionTree':      DecisionTreeClassifier(max_depth=25, random_state=0),
}
print(f'{"model":34s} {"CV macro-F1":>12} {"+/- SE":>8}')
for name, c_ in contenders.items():
    t0 = time.time()
    sc = cross_val_score(pipe(c_), texts, y, cv=5, scoring='f1_macro', n_jobs=-1)
    print(f'{name:34s} {sc.mean():12.3f} {sc.std(ddof=1)/np.sqrt(5):8.3f}   ({time.time()-t0:.0f}s)')

**Read the bake-off:** the ranking is *evidence* — same folds, same metric, SEs
attached; a gap under ~2 SE is a coin flip, and saying so out loud will own the room.
Each type has a personality (ComplementNB = imbalance-robust counting, built for skewed
text; LinearSVC = max-margin linear; logistic = the calibrated workhorse — here with
`class_weight='balanced'`, the imbalance crash course's one-keyword fix; the tree =
interpretable axis-splits, usually outclassed on sparse text). And the representation
gap — presence features vs tuned TF-IDF — is your **BERT foreshadow**: better features
beat better knobs.

## The finale: leakage, or free accuracy on coin flips

Labels below are **literal coin flips** — nothing to learn. Select features on ALL the
data first, and CV happily validates garbage, because the held-out fold's noise voted on
which columns survived.

*(Verified with this exact protocol in Briefing 1: leaky ≈ **0.815 ± 0.051**, honest
Pipeline ≈ 0.515 ± 0.064 — Freedman 1983; Ambroise & McLachlan 2002.)*

In [ ]:
r7 = np.random.default_rng(7)
leaky, honest = [], []
for rep in range(10):
    Xn = r7.normal(size=(400, 1000))
    yn_ = r7.integers(0, 2, 400)

    sel = SelectKBest(lambda A, b: (np.abs((A - A.mean(0)).T @ (b - b.mean())), None),
                      k=20).fit(Xn, yn_)               # SIN: fit on ALL rows
    leaky.append(cross_val_score(LogisticRegression(), sel.transform(Xn), yn_,
                                 cv=5).mean())

    honest_pipe = Pipeline([('sel', SelectKBest(
                        lambda A, b: (np.abs((A - A.mean(0)).T @ (b - b.mean())), None), k=20)),
                            ('clf', LogisticRegression())])
    honest.append(cross_val_score(honest_pipe, Xn, yn_, cv=5).mean())

print('labels are COIN FLIPS — true attainable accuracy: 0.500\n')
print(f'  select-then-CV (leaky):     {np.mean(leaky):.3f} +/- {np.std(leaky):.3f}')
print(f'  Pipeline (select in folds): {np.mean(honest):.3f} +/- {np.std(honest):.3f}')

fig, ax = plt.subplots(figsize=(5.2, 3.2))
ax.bar(['leaky\n(select, then CV)', 'honest\n(Pipeline in CV)'],
       [np.mean(leaky), np.mean(honest)],
       yerr=[np.std(leaky), np.std(honest)], capsize=6, width=.5)
ax.axhline(0.5, ls='--', color='k', lw=1, label='truth: coin flip')
ax.set_ylabel('CV accuracy'); ax.set_ylim(.4, .95); ax.legend()
ax.set_title('Pure noise. One protocol lies.')
plt.tight_layout(); plt.show()

**The rule is absolute: every data-dependent step — selection, scaling, resampling —
lives inside the fold.** In sklearn that is one idiom: build a `Pipeline`,
cross-validate the `Pipeline`. (Real published science has died of the leaky version.)

> **🎤 Act IV, what you say on stage:** *"Every knob so far, I chose in front of you by
> hand. Here's the honest machinery: macro-F1, because every hazard class gets one vote;
> cross-validation — just a for-loop, five train/judge rounds, error bars for free; the
> one-SE rule to prefer the simplest model within noise of the best; then a bake-off of
> model types on the same folds, picked on evidence. And the commandment that makes it
> all valid: every data-dependent step lives inside the fold. Break it and you conjure
> accuracy out of literal coin flips — 0.815 on pure noise, verified. In sklearn the fix
> is one idiom: Pipeline, then cross-validate the Pipeline."*

---
# Epilogue

Act II ends owing *"can the model select for itself?"* → Act III's L1 pays it.
Act III ends owing *"who picks λ?"* → Act IV's CV pays it. Act IV ends owing *"better
features beat better knobs"* → your BERT close pays it.

| claim | where verified |
| --- | --- |
| monoxide/fire chi²-vs-MI inversion | Briefing 1 §2; both notebooks |
| 6,907 → 1,000 features costs 0.001 macro-F1 | Briefing 1 §5 |
| bias²+var = MSE; U-curve; leash halves error | Briefing 2 §3.3; both notebooks |
| L1 dial (C=0.05 → 12 words F1 .72; C=1 → 278, .87) | Briefing 1 §6 (this API) |
| tuning plateau; one-SE → λ=10 | Briefing 3/4 §3; both notebooks |
| bake-off protocol; TF-IDF pipeline ≈ .856 | Briefing 3/4 §4; Briefing 1 |
| leakage 0.815 vs 0.515 on coin flips | Briefing 1 §8; this notebook |

**Pre-talk checklist still open:** workbook Run All · `class_weight` delta (imbalance
crash course §6) · AdamW decay from `*_meta.txt` · colleague's BERT numbers
(probe or fine-tune?).